# Anotador TDAH · 04/04 · Backend `instructor`

Usa la librería `instructor` sobre el endpoint OpenAI-compatible de Ollama (`/v1`). La salida se valida contra el modelo Pydantic `Anotacion` y, si la validación falla, **reintenta automáticamente** (hasta 2 veces) enviando el error de validación al modelo para que se corrija.

Es el backend es el mas robusto, pero mas penañizado en latencia a cambio de garantizar que la salida cumple el esquema con tipos validados.


## 1 · Parámetros

In [1]:
import datetime as dt
import json
import sqlite3
import time

import pandas as pd

SEMANA       = 1            # semana de seguimiento (el dataset llega a la 24)
PACIENTES    = None         # None = todos los de la semana; o lista: ["P001", "P003"]
REPETICIONES = 3            # veces que se anota cada entrada
TEMPERATURA  = 0.7
MODELO       = "gemma4:e4b"

# --- Rutas y conexión ---
RUTA_BD     = "datos/anotador.db"
OLLAMA_URL  = "http://127.0.0.1:11002"   
INSTRUMENTO = "instrumentos/brief2.json"

BACKEND     = "instructor"
EXPERIMENTO = f"{BACKEND}-s{SEMANA}-t{TEMPERATURA}-{dt.date.today():%Y%m%d}"
print(f"Código de experimento: {EXPERIMENTO}")

Código de experimento: instructor-s1-t0.7-20260713


## 2 · Datos

Las entradas (texto libre de los padres) de la semana elegida, con el contexto del paciente (edad calculada a la fecha de la observación, sexo, quién informa).

In [2]:
instrumento = json.load(open(INSTRUMENTO, encoding="utf-8"))
print(f"Instrumento: {instrumento['nombre']} ({len(instrumento['items'])} ítems)")

con = sqlite3.connect(RUTA_BD)

entradas = pd.read_sql(
    '''
    SELECT e.id_entrada, e.id_paciente, c.rol AS informante, e.fecha,
           p.fecha_nacimiento, p.sexo, e.texto
    FROM entrada e
    JOIN paciente p USING (id_paciente)
    JOIN cuidador c ON c.id_cuidador = e.id_cuidador
    JOIN referencia_sintetica r USING (id_entrada)
    WHERE r.semana = ?
    ORDER BY e.id_paciente
    ''',
    con, params=[SEMANA],
)
if PACIENTES:
    entradas = entradas[entradas["id_paciente"].isin(PACIENTES)]


def calcular_edad(nacimiento, observacion):
    nac = pd.to_datetime(nacimiento).date()
    obs = pd.to_datetime(observacion).date()
    return obs.year - nac.year - ((obs.month, obs.day) < (nac.month, nac.day))


entradas["edad"] = [
    calcular_edad(n, f) for n, f in zip(entradas["fecha_nacimiento"], entradas["fecha"])
]

print(f"Semana {SEMANA}: {len(entradas)} entradas de {entradas['id_paciente'].nunique()} pacientes")
entradas[["id_entrada", "id_paciente", "informante", "edad", "sexo", "texto"]].head()

Instrumento: BRIEF-2 Familia (63 ítems)
Semana 1: 30 entradas de 30 pacientes


,id_entrada,id_paciente,informante,edad,sexo,texto
0,1,PAC001,madre,7,masculino,"Hoy ha sido un día horrible, la verdad. Marco ..."
1,25,PAC002,madre,11,femenino,"Hola, soy la madre de Lucía. Nos dijeron que t..."
2,49,PAC003,padre,15,masculino,Soy el padre de Alejandro. La psiquiatra nos h...
3,73,PAC004,madre,6,femenino,"Somos los padres de Sofía, tiene 6 años. Esta ..."
4,97,PAC005,madre,8,masculino,Soy la madre de Diego. Diego vive conmigo de l...


## 3 · Prompts

El prompt de sistema se construye desde el instrumento (`brief2.json`): catálogo de ítems, escalas y niveles de alerta. El de usuario lleva el contexto del paciente y su texto.

In [3]:
COMILLAS = '"' * 3  

def construir_prompt_sistema(instrumento):
    catalogo = "\n".join(
        f"  {it['id']}: [{it['escala']}] {it['texto']}" for it in instrumento["items"]
    )
    escalas = "\n".join(f"  - {e}: {d}" for e, d in instrumento["escalas"].items())
    n = instrumento["niveles_alerta"]
    return f'''Eres un {instrumento["rol_anotador"]}.

Tu tarea es analizar el texto libre de observación de un padre/madre sobre su hijo/a
y producir una anotación clínica estructurada en formato JSON, basada en el instrumento
{instrumento["nombre"]}.

## CATÁLOGO DE ÍTEMS ({len(instrumento["items"])} ítems)
{catalogo}

## ESCALAS
{escalas}

## NIVELES DE ALERTA
{" | ".join(n)}

## INSTRUCCIONES DE SALIDA
Responde ÚNICAMENTE con un objeto JSON válido, sin texto antes ni después, sin markdown.
Estructura requerida:
{{
  "items_detectados": [lista de números de ítem observables en el texto],
  "escalas_afectadas": [lista de escalas correspondientes],
  "nivel_alerta": "{n[0]}|{n[1]}|{n[2]}",
  "nota_clinica": "resumen clínico de 1-3 frases para el médico",
  "justificacion": "explicación del razonamiento (para auditoría)"
}}'''


def construir_prompt_usuario(e):
    return f'''## CONTEXTO DEL PACIENTE
- Edad: {e.edad} años
- Sexo: {e.sexo}
- Informante: {e.informante}

## TEXTO DEL PADRE/MADRE
{COMILLAS}{e.texto}{COMILLAS}

Analiza el texto y genera el JSON de anotación clínica.'''


prompt_sistema = construir_prompt_sistema(instrumento)
print(prompt_sistema[:400] + "\n[...]")

Eres un psicólogo clínico infantil especializado en TDAH y en el instrumento BRIEF-2.

Tu tarea es analizar el texto libre de observación de un padre/madre sobre su hijo/a
y producir una anotación clínica estructurada en formato JSON, basada en el instrumento
BRIEF-2 Familia.

## CATÁLOGO DE ÍTEMS (63 ítems)
  1: [inhibicion] Es inquieto o inquieta.
  2: [flexibilidad] Se resiste o le cuesta acept
[...]


## 4 · Backend Instructor


In [4]:
from pydantic import BaseModel, Field


class Anotacion(BaseModel):
    items_detectados: list[int] = Field(default_factory=list)
    escalas_afectadas: list[str] = Field(default_factory=list)
    nivel_alerta: str = "bajo"
    nota_clinica: str = ""
    justificacion: str = ""

In [5]:
import instructor
from openai import OpenAI

cliente = instructor.from_openai(
    OpenAI(base_url=f"{OLLAMA_URL}/v1", api_key="ollama"),
    mode=instructor.Mode.JSON,
)


def anotar(prompt_sistema, prompt_usuario):
    '''Llama al modelo y devuelve (anotacion | None, respuesta_cruda).'''
    try:
        obj = cliente.chat.completions.create(
            model=MODELO,
            response_model=Anotacion,
            max_retries=2,
            temperature=TEMPERATURA,
            messages=[
                {"role": "system", "content": prompt_sistema},
                {"role": "user", "content": prompt_usuario},
            ],
        )
    except Exception as e:
        return None, f"error: {e}"
    datos = obj.model_dump()
    return datos, json.dumps(datos, ensure_ascii=False)

## 5 · Una anotación de ejemplo

Antes de lanzar el experimento completo, una sola entrada para ver la anotación final que produce este backend.

In [ ]:
ejemplo = entradas.iloc[0]
print(f"Paciente {ejemplo.id_paciente} · {ejemplo.informante} · semana {SEMANA}")
print(f"Texto: {ejemplo.texto[:200]}...\n")

t0 = time.time()
anotacion, cruda = anotar(prompt_sistema, construir_prompt_usuario(ejemplo))
print(f"Latencia: {time.time() - t0:.1f}s\n")

if anotacion is None:
    print("[FALLO DE FORMATO] El modelo no devolvió un JSON válido:")
    print(cruda[:500])
else:
    print("ANOTACIÓN FINAL:")
    print(json.dumps(anotacion, indent=2, ensure_ascii=False))

## 6 · El experimento

Anota cada entrada de la semana `REPETICIONES` veces y guarda cada resultado en la tabla `experimento` con el código `EXPERIMENTO`. Repetir la misma entrada es lo que permite medir si el modelo es estable.

> Si relanzas con el mismo código se añaden filas al mismo experimento. Para empezar de cero: `con.execute("DELETE FROM experimento WHERE codigo = ?", [EXPERIMENTO]); con.commit()`

In [6]:
con.execute('''
CREATE TABLE IF NOT EXISTS experimento (
    id                INTEGER PRIMARY KEY,
    codigo            TEXT NOT NULL,      -- código del experimento (para comparar)
    creada_en         TEXT NOT NULL,
    backend           TEXT NOT NULL,
    modelo            TEXT NOT NULL,
    temperatura       REAL NOT NULL,
    semana            INTEGER,
    id_paciente       TEXT,
    id_entrada        INTEGER,
    repeticion        INTEGER,
    formato_ok        INTEGER,
    items_detectados  TEXT,               -- JSON: [int]
    escalas_afectadas TEXT,               -- JSON: [str]
    nivel_alerta      TEXT,
    nota_clinica      TEXT,
    justificacion     TEXT,
    latencia_s        REAL
)''')
con.commit()

total = len(entradas) * REPETICIONES
print(f"Experimento '{EXPERIMENTO}': {len(entradas)} entradas × {REPETICIONES} repeticiones "
      f"= {total} llamadas al modelo")

hechas = 0
for _, e in entradas.iterrows():
    prompt_usuario = construir_prompt_usuario(e)
    for rep in range(REPETICIONES):
        t0 = time.time()
        anotacion, cruda = anotar(prompt_sistema, prompt_usuario)
        latencia = time.time() - t0
        ok = anotacion is not None
        a = anotacion or {}
        con.execute(
            "INSERT INTO experimento (codigo, creada_en, backend, modelo, temperatura, "
            "semana, id_paciente, id_entrada, repeticion, formato_ok, items_detectados, "
            "escalas_afectadas, nivel_alerta, nota_clinica, justificacion, latencia_s) "
            "VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
            (EXPERIMENTO, dt.datetime.now().isoformat(timespec="seconds"), BACKEND,
             MODELO, TEMPERATURA, SEMANA, e.id_paciente, int(e.id_entrada), rep,
             int(ok), json.dumps(a.get("items_detectados", [])),
             json.dumps(a.get("escalas_afectadas", [])), a.get("nivel_alerta"),
             a.get("nota_clinica"), a.get("justificacion"), latencia),
        )
        con.commit()
        hechas += 1
        estado = "ok" if ok else "FALLO DE FORMATO"
        print(f"  [{hechas:>3}/{total}] {e.id_paciente} rep {rep + 1} → {estado} ({latencia:.1f}s)")

print(f"\nGuardado en la tabla `experimento` con codigo = '{EXPERIMENTO}'")

Experimento 'instructor-s1-t0.7-20260713': 30 entradas × 3 repeticiones = 90 llamadas al modelo
  [  1/90] PAC001 rep 1 → ok (18.7s)
  [  2/90] PAC001 rep 2 → ok (12.3s)
  [  3/90] PAC001 rep 3 → ok (11.5s)
  [  4/90] PAC002 rep 1 → ok (8.9s)
  [  5/90] PAC002 rep 2 → ok (10.5s)
  [  6/90] PAC002 rep 3 → ok (9.3s)
  [  7/90] PAC003 rep 1 → ok (8.7s)
  [  8/90] PAC003 rep 2 → ok (10.0s)
  [  9/90] PAC003 rep 3 → ok (8.7s)
  [ 10/90] PAC004 rep 1 → ok (11.2s)
  [ 11/90] PAC004 rep 2 → ok (10.5s)
  [ 12/90] PAC004 rep 3 → ok (11.1s)
  [ 13/90] PAC005 rep 1 → ok (10.5s)
  [ 14/90] PAC005 rep 2 → ok (11.1s)
  [ 15/90] PAC005 rep 3 → ok (11.2s)
  [ 16/90] PAC006 rep 1 → ok (8.5s)
  [ 17/90] PAC006 rep 2 → ok (8.3s)
  [ 18/90] PAC006 rep 3 → ok (7.8s)
  [ 19/90] PAC007 rep 1 → ok (8.3s)
  [ 20/90] PAC007 rep 2 → ok (13.5s)
  [ 21/90] PAC007 rep 3 → ok (13.7s)
  [ 22/90] PAC008 rep 1 → ok (8.7s)
  [ 23/90] PAC008 rep 2 → ok (8.0s)
  [ 24/90] PAC008 rep 3 → ok (8.3s)
  [ 25/90] PAC009 rep 1 → o

## 7 · Resultados

- `formato_ok`: fracción de salidas que fueron JSON válido.
- `acuerdo_nivel` (0–1): fracción de repeticiones que coincide con el nivel de alerta más frecuente de ese paciente. 1.0 = el modelo dice siempre lo mismo.
- `latencia_media`: segundos por anotación.

In [7]:
df = pd.read_sql(
    "SELECT * FROM experimento WHERE codigo = ?", con, params=[EXPERIMENTO]
)
print(f"{len(df)} anotaciones del experimento '{EXPERIMENTO}'\n")


def acuerdo_modal(niveles):
    '''Fracción de repeticiones que coincide con el nivel más frecuente.'''
    s = niveles.dropna()
    return round(s.value_counts().iloc[0] / len(s), 2) if len(s) else None


resumen = df.groupby("id_paciente").agg(
    repeticiones=("repeticion", "count"),
    formato_ok=("formato_ok", "mean"),
    acuerdo_nivel=("nivel_alerta", acuerdo_modal),
    latencia_media=("latencia_s", "mean"),
).round(2)

print(f"Formato válido: {df['formato_ok'].mean():.0%}")
print(f"Acuerdo medio del nivel de alerta entre repeticiones: {resumen['acuerdo_nivel'].mean():.2f}")
print(f"Latencia media por anotación: {df['latencia_s'].mean():.1f}s\n")
resumen

90 anotaciones del experimento 'instructor-s1-t0.7-20260713'

Formato válido: 100%
Acuerdo medio del nivel de alerta entre repeticiones: 0.86
Latencia media por anotación: 9.1s



,repeticiones,formato_ok,acuerdo_nivel,latencia_media
id_paciente,,,,
PAC001,3,1.0,0.67,14.15
PAC002,3,1.0,0.67,9.57
PAC003,3,1.0,0.67,9.14
PAC004,3,1.0,0.67,10.93
PAC005,3,1.0,1.00,10.92
PAC006,3,1.0,1.00,8.22
PAC007,3,1.0,0.67,11.80
PAC008,3,1.0,1.00,8.35
PAC009,3,1.0,1.00,8.03


In [8]:
# Las anotaciones de un paciente concreto, repetición a repetición
UN_PACIENTE = df["id_paciente"].iloc[0]   # cambiar por el que interese

detalle = df[df["id_paciente"] == UN_PACIENTE]
for _, fila in detalle.iterrows():
    print(f"— repetición {fila.repeticion}: nivel={fila.nivel_alerta} "
          f"items={fila.items_detectados}")
    print(f"  nota: {fila.nota_clinica}\n")

— repetición 0: nivel=alto items=[1, 4, 3]
  nota: El niño muestra dificultad en la regulación motora (saltar constantemente) y presenta fallos notables en la memoria de trabajo al seguir instrucciones múltiples. Además, su conducta es impulsiva e impacta negativamente a sus pares. Estos patrones sugieren un impacto significativo del TDAH que requiere intervención multimodal.

— repetición 1: nivel=alto items=[1, 3, 4, 10]
  nota: Se observan dificultades significativas en la regulación de la conducta e impulsividad (1, 10). La madre reporta problemas con la memoria de trabajo al intentar ejecutar tareas secuenciales (3), además de déficits en la conciencia del impacto social de sus acciones (4). Estos patrones sugieren un perfil sintomático que amerita una evaluación exhaustiva y apoyo parental intensivo.

— repetición 2: nivel=moderado items=[1, 3, 4, 9]
  nota: El niño presenta dificultades evidentes en la regulación de su conducta motora (saltar, correr) y problemas significativos 

## 8 · Siguiente paso

Comparar este experimento con los de los otros backends (u otros parámetros) en `05_comparacion_experimentos.ipynb`, usando los códigos de experimento.